# Flask + ngrok demo

Run the cells top-to-bottom to:
1) start a local Flask app
2) expose it to the internet via ngrok

If ngrok asks for an authtoken, set `NGROK_AUTHTOKEN` in your environment (or paste it into the cell).

In [1]:
from kaggle_secrets import UserSecretsClient
import os


user_secrets = UserSecretsClient()
#secret_value_0 = user_secrets.get_secret("db_string")
os.environ["HF_TOKEN"] = hf_token = user_secrets.get_secret("HF_TOKEN")
#secret_value_2 = user_secrets.get_secret("mega_email")
#secret_value_3 = user_secrets.get_secret("mega_pass")
#secret_value_4 = user_secrets.get_secret("ngrok")
ngrok_2 = user_secrets.get_secret("ngrok2")



os.environ["NGROK_AUTHTOKEN"] = user_secrets.get_secret("ngrok2")

In [2]:
isCondaSet = False
isDA3Set = False

In [3]:
# (Optional) Install deps into the active Jupyter kernel environment
import sys

!{sys.executable} -m pip install -q flask pyngrok gdown mega.py imageio-ffmpeg opencv-python-headless

In [4]:
from flask import Flask, jsonify, render_template_string, request, send_from_directory, url_for
from threading import Thread, Lock, Event
from werkzeug.serving import make_server
from werkzeug.utils import secure_filename
from pathlib import Path
from collections import deque
from datetime import datetime
from queue import Queue, Empty, Full
import os
import shutil
import subprocess
import ipaddress
import socket
import time
import itertools
import urllib.parse
import urllib.request
import uuid
import cv2


TWELVE_HOURS_SEC = 12 * 60 * 60


def _get_process_start_time_epoch():
    # Best-effort kernel start timestamp (seconds since Unix epoch)
    try:
        import psutil  # type: ignore

        return float(psutil.Process(os.getpid()).create_time())
    except Exception:
        pass

    if os.name == "nt":
        try:
            import ctypes
            import ctypes.wintypes as wintypes

            class FILETIME(ctypes.Structure):
                _fields_ = [
                    ("dwLowDateTime", wintypes.DWORD),
                    ("dwHighDateTime", wintypes.DWORD),
                ]

            creation_time = FILETIME()
            exit_time = FILETIME()
            kernel_time = FILETIME()
            user_time = FILETIME()

            kernel32 = ctypes.WinDLL("kernel32", use_last_error=True)
            kernel32.GetCurrentProcess.restype = wintypes.HANDLE
            kernel32.GetProcessTimes.argtypes = [
                wintypes.HANDLE,
                ctypes.POINTER(FILETIME),
                ctypes.POINTER(FILETIME),
                ctypes.POINTER(FILETIME),
                ctypes.POINTER(FILETIME),
            ]
            kernel32.GetProcessTimes.restype = wintypes.BOOL

            handle = kernel32.GetCurrentProcess()
            ok = kernel32.GetProcessTimes(
                handle,
                ctypes.byref(creation_time),
                ctypes.byref(exit_time),
                ctypes.byref(kernel_time),
                ctypes.byref(user_time),
            )
            if not ok:
                raise ctypes.WinError(ctypes.get_last_error())

            ft = (creation_time.dwHighDateTime << 32) + creation_time.dwLowDateTime
            return (ft / 10_000_000) - 11644473600
        except Exception:
            pass

    return None


def _format_hhmm(seconds: float) -> str:
    seconds = max(0, int(seconds))
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    return f"{hours:02d}:{minutes:02d}"


# --- Web terminal (call web_print(...) from other cells) ---
if "WEB_PRINT_QUEUE" not in globals():
    WEB_PRINT_QUEUE = Queue(maxsize=5000)
if "WEB_TERMINAL_BUFFER" not in globals():
    WEB_TERMINAL_BUFFER = deque(maxlen=2000)
if "WEB_TERMINAL_LOCK" not in globals():
    WEB_TERMINAL_LOCK = Lock()


def web_print(msg):
    ts = datetime.now().strftime("%H:%M:%S")
    text = "" if msg is None else str(msg)
    lines = text.splitlines() or [""]
    for line in lines:
        entry = f"[{ts}] {line}"
        try:
            WEB_PRINT_QUEUE.put_nowait(entry)
        except Full:
            try:
                WEB_PRINT_QUEUE.get_nowait()
            except Empty:
                pass
            try:
                WEB_PRINT_QUEUE.put_nowait(entry)
            except Exception:
                pass


def _drain_web_print_queue(max_items: int = 500) -> int:
    drained = 0
    while drained < max_items:
        try:
            entry = WEB_PRINT_QUEUE.get_nowait()
        except Empty:
            break
        with WEB_TERMINAL_LOCK:
            WEB_TERMINAL_BUFFER.append(entry)
        drained += 1
    return drained


# --- Uploaded video queue (FIFO) ---
VIDEO_EXTENSIONS = {
    ".mp4",
    ".mov",
    ".mkv",
    ".avi",
    ".webm",
    ".flv",
    ".wmv",
    ".m4v",
    ".3gp",
    ".mpeg",
    ".mpg",
    ".dav",
}

if "VIDEO_UPLOAD_QUEUE" not in globals():
    VIDEO_UPLOAD_QUEUE = deque(maxlen=200)
if "VIDEO_QUEUE_LOCK" not in globals():
    VIDEO_QUEUE_LOCK = Lock()

# --- fps adjusted queue (prepared videos) ---
if "DA3_QUEUE" not in globals():
    DA3_QUEUE = deque(maxlen=200)
if "DA3_QUEUE_LOCK" not in globals():
    DA3_QUEUE_LOCK = Lock()
if "DA3_JOB_QUEUE" not in globals():
    DA3_JOB_QUEUE = Queue(maxsize=500)
if "DA3_SEEN" not in globals():
    DA3_SEEN = set()
if "DA3_SEEN_LOCK" not in globals():
    DA3_SEEN_LOCK = Lock()

# --- Frame queue (bounded frames on disk) ---
if "FRAME_QUEUE" not in globals():
    FRAME_QUEUE = deque(maxlen=100)
if "FRAME_QUEUE_LOCK" not in globals():
    FRAME_QUEUE_LOCK = Lock()
if "FRAME_QUEUE_FILL_LOCK" not in globals():
    FRAME_QUEUE_FILL_LOCK = Lock()
if "FRAME_VIDEO_STATE" not in globals():
    FRAME_VIDEO_STATE = {}
if "FRAME_VIDEO_STATE_LOCK" not in globals():
    FRAME_VIDEO_STATE_LOCK = Lock()
if "FRAME_VIDEO_ORDER" not in globals():
    FRAME_VIDEO_ORDER = deque()
if "FRAME_QUEUE_WAKE_EVENT" not in globals():
    FRAME_QUEUE_WAKE_EVENT = Event()
if "FRAME_QUEUE_STOP_EVENT" not in globals():
    FRAME_QUEUE_STOP_EVENT = Event()
if "FRAME_QUEUE_WORKER" not in globals():
    FRAME_QUEUE_WORKER = None


def _is_video_path(path: Path) -> bool:
    # Prefer extension check; fall back to lightweight header sniffing.
    if path.suffix.lower() in VIDEO_EXTENSIONS:
        return True

    try:
        with open(path, "rb") as f:
            head = f.read(64)
    except Exception:
        return False

    # MP4/MOV/3GP family: ....ftyp....
    if len(head) >= 8 and head[4:8] == b"ftyp":
        return True
    # Matroska/WebM
    if head.startswith(bytes.fromhex("1a45dfa3")):
        return True
    # AVI
    if len(head) >= 12 and head.startswith(b"RIFF") and head[8:12] in {b"AVI ", b"AVIX"}:
        return True
    # FLV
    if head.startswith(b"FLV"):
        return True
    # Ogg container (may include video)
    if head.startswith(b"OggS"):
        return True
    # ASF (WMV)
    if head.startswith(bytes.fromhex("3026b2758e66cf11a6d900aa0062ce6c")):
        return True
    # MPEG program/elementary stream (very rough)
    if head[:4] in {bytes.fromhex("000001ba"), bytes.fromhex("000001b3")}:
        return True

    return False


def _ffmpeg_exe() -> str:
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    try:
        import imageio_ffmpeg  # type: ignore

        exe = imageio_ffmpeg.get_ffmpeg_exe()
        if exe:
            return exe
    except Exception:
        pass
    raise RuntimeError("ffmpeg not found. Install ffmpeg and ensure it's on PATH.")


def convert_dav_to_mp4(dav_path: Path) -> Path:
    if dav_path.suffix.lower() != ".dav":
        return dav_path

    out_path = dav_path.with_suffix(".mp4")
    if out_path.exists():
        out_path = dav_path.with_name(f"{dav_path.stem}_{uuid.uuid4().hex[:8]}.mp4")

    ffmpeg = _ffmpeg_exe()
    attempts = [
        ("video+audio", ["-map", "0:v:0", "-map", "0:a?", "-c", "copy"]),
        ("video-only", ["-map", "0:v:0", "-c", "copy"]),
    ]

    last_err = ""
    for label, extra in attempts:
        out_path.unlink(missing_ok=True)
        cmd = [
            ffmpeg,
            "-y",
            "-hide_banner",
            "-loglevel",
            "error",
            "-fflags",
            "+genpts",
            "-i",
            str(dav_path),
            *extra,
            "-movflags",
            "+faststart",
            str(out_path),
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode == 0 and out_path.exists() and out_path.stat().st_size > 0:
            web_print(f"DAV->MP4 conversion ok ({label}): {out_path.name}")
            return out_path

        last_err = (proc.stderr or proc.stdout or "").strip()
        if len(last_err) > 3000:
            last_err = last_err[-3000:]

    out_path.unlink(missing_ok=True)
    raise RuntimeError(f"ffmpeg DAV->MP4 failed. {last_err}")

    return out_path


def convert_video_to_target_fps(src_path: Path, target_fps: int) -> Path:
    fps = int(target_fps)
    if fps < 1 or fps > 240:
        raise ValueError('Target FPS must be between 1 and 240.')

    base = secure_filename(src_path.stem) or 'video'
    base = base[:60]
    out_name = f'fpsadj_{uuid.uuid4().hex[:12]}_{base}_fps{fps}.mp4'
    out_path = UPLOAD_DIR / out_name

    ffmpeg = _ffmpeg_exe()
    attempts = [
        ('copy-audio', ['-c:a', 'copy']),
        ('aac-audio', ['-c:a', 'aac', '-b:a', '192k']),
    ]

    last_err = ''
    for label, audio_args in attempts:
        out_path.unlink(missing_ok=True)
        cmd = [
            ffmpeg,
            '-y',
            '-hide_banner',
            '-loglevel',
            'error',
            '-fflags',
            '+genpts',
            '-i',
            str(src_path),
            '-map',
            '0:v:0',
            '-map',
            '0:a?',
            '-vf',
            f'fps={fps}',
            '-c:v',
            'libx264',
            '-preset',
            'veryfast',
            '-crf',
            '0',
            *audio_args,
            '-movflags',
            '+faststart',
            str(out_path),
        ]
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode == 0 and out_path.exists() and out_path.stat().st_size > 0:
            web_print(f'FPS adjust ok ({label}): {out_path.name}')
            return out_path

        last_err = (proc.stderr or proc.stdout or '').strip()
        if len(last_err) > 3000:
            last_err = last_err[-3000:]

    out_path.unlink(missing_ok=True)
    raise RuntimeError(f'ffmpeg fps conversion failed. {last_err}')


def enqueue_da3_job(video_path: Path, source: str = '') -> None:
    fps = get_target_fps()
    key = (str(video_path), int(fps))
    with DA3_SEEN_LOCK:
        if key in DA3_SEEN:
            return
        DA3_SEEN.add(key)

    job = {
        'input_path': str(video_path),
        'input_stored_name': video_path.name,
        'target_fps': int(fps),
        'ts': float(time.time()),
        'source': str(source or ''),
    }
    try:
        DA3_JOB_QUEUE.put_nowait(job)
    except Full:
        with DA3_SEEN_LOCK:
            DA3_SEEN.discard(key)
        web_print(f'FPS adjust job queue full; dropped: {video_path.name}')
        return

    web_print(f'FPS adjust job queued: {video_path.name} -> {fps}fps')


def _da3_worker_loop():
    web_print('FPS adjust worker started')
    while True:
        try:
            job = DA3_JOB_QUEUE.get(timeout=1)
        except Empty:
            continue
        if job is None:
            break

        src_path = Path(job.get('input_path', ''))
        fps = int(job.get('target_fps', 15))
        try:
            out_path = convert_video_to_target_fps(src_path, fps)
            try:
                size_bytes = out_path.stat().st_size
            except Exception:
                size_bytes = 0

            item = {
                'stored_name': out_path.name,
                'path': str(out_path),
                'size_bytes': int(size_bytes),
                'size_mb': round(size_bytes / (1024 * 1024), 2),
                'ts': float(time.time()),
                'source': job.get('source', ''),
                'target_fps': fps,
                'input_stored_name': job.get('input_stored_name', src_path.name),
                'input_path': job.get('input_path', str(src_path)),
            }

            dropped = None
            with DA3_QUEUE_LOCK:
                if DA3_QUEUE.maxlen and len(DA3_QUEUE) >= DA3_QUEUE.maxlen:
                    dropped = DA3_QUEUE[0]
                DA3_QUEUE.append(item)

            if dropped:
                web_print(f"FPS adjusted queue full; dropped oldest: {dropped.get('stored_name')}")
            web_print(f"FPS adjusted queued: {out_path.name}")

            try:
                register_video_for_frame_queue(out_path, source=job.get('source', ''))
            except Exception as exc:
                web_print(f"Frame queue register failed for {out_path.name}: {exc}")
        except Exception as exc:
            web_print(f"FPS adjust failed for {src_path.name}: {exc}")


def start_da3_worker():
    global DA3_WORKER
    try:
        if DA3_WORKER is not None and DA3_WORKER.is_alive():
            return
    except Exception:
        pass
    DA3_WORKER = Thread(target=_da3_worker_loop, daemon=True, name='FPS-Adjust-Worker')
    DA3_WORKER.start()


def da3_queue_list():
    with DA3_QUEUE_LOCK:
        return list(DA3_QUEUE)


def da3_queue_pop():
    with DA3_QUEUE_LOCK:
        if not DA3_QUEUE:
            return None
        return DA3_QUEUE.popleft()


def _is_da3_set() -> bool:
    try:
        return bool(globals().get("isDA3Set", False))
    except Exception:
        return False


def register_video_for_frame_queue(video_path: Path, video_stored_name: str = "", source: str = "") -> None:
    if not video_stored_name:
        video_stored_name = video_path.name
    key = str(video_path.resolve())

    with FRAME_VIDEO_STATE_LOCK:
        if key in FRAME_VIDEO_STATE:
            return

        base = secure_filename(video_path.stem) or 'video'
        base = base[:60]
        digest = uuid.uuid5(uuid.NAMESPACE_URL, key).hex[:10]
        frames_dir = FRAMES_DIR / f'{base}_{digest}'

        FRAME_VIDEO_STATE[key] = {
            'video_path': key,
            'video_stored_name': str(video_stored_name),
            'video_file_url': f'/uploads/{video_stored_name}',
            'frames_dir': str(frames_dir),
            'next_frame_idx': 0,
            'frame_count': None,
            'report_every': 100,
            'next_report': 100,
            'ts': float(time.time()),
            'source': str(source or ''),
        }
        FRAME_VIDEO_ORDER.append(key)

    web_print(f'Frame queue: registered video: {video_path.name}')
    start_frame_queue_worker()
    FRAME_QUEUE_WAKE_EVENT.set()


def _opencv_extract_frame(video_path: Path, frame_idx: int, out_path: Path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise RuntimeError('OpenCV failed to open video')

    try:
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        if frame_count <= 0:
            frame_count = None

        cap.set(cv2.CAP_PROP_POS_FRAMES, float(frame_idx))
        ok, frame = cap.read()
    finally:
        cap.release()

    if not ok or frame is None:
        return False, frame_count

    out_path.parent.mkdir(parents=True, exist_ok=True)
    ok2 = cv2.imwrite(str(out_path), frame, [cv2.IMWRITE_PNG_COMPRESSION, 3])
    if not ok2:
        raise RuntimeError('Failed to write frame image')
    return True, frame_count


def _next_frame_task_for_video(video_key: str):
    with FRAME_VIDEO_STATE_LOCK:
        state = FRAME_VIDEO_STATE.get(video_key)
        if not state:
            return None
        video_path = Path(state.get('video_path', ''))
        video_stored_name = state.get('video_stored_name', '') or video_path.name
        frames_dir = Path(state.get('frames_dir', ''))
        next_idx = int(state.get('next_frame_idx', 0))
        frame_count = state.get('frame_count', None)

    if not video_path.exists():
        raise FileNotFoundError(f'Video not found: {video_path}')

    if frame_count is not None and next_idx >= int(frame_count):
        return None

    out_path = frames_dir / f'frame_{next_idx:06d}.png'
    if not out_path.exists():
        ok, fc = _opencv_extract_frame(video_path, next_idx, out_path)
        if fc is not None:
            frame_count = int(fc)
        if not ok:
            with FRAME_VIDEO_STATE_LOCK:
                st = FRAME_VIDEO_STATE.get(video_key)
                if st is not None and frame_count is not None:
                    st['frame_count'] = int(frame_count)
                    st['done'] = True
            return None

    rel = out_path.relative_to(FRAMES_DIR).as_posix()
    task = {
        'video_file_url': f'/uploads/{video_stored_name}',
        'frame_no': int(next_idx),
        'frame_download_url': f'/frames/{rel}',
        'frame_local_path': str(out_path.resolve()),
    }

    msg = None
    with FRAME_VIDEO_STATE_LOCK:
        st = FRAME_VIDEO_STATE.get(video_key)
        if st is not None:
            if frame_count is not None:
                st['frame_count'] = int(frame_count)
                if st.get('report_every', 100) == 100:
                    rep = max(10, int(frame_count) // 20) if int(frame_count) else 100
                    st['report_every'] = int(rep)
                    st['next_report'] = int(rep)

            st['next_frame_idx'] = int(next_idx) + 1
            extracted = int(st['next_frame_idx'])
            fc2 = st.get('frame_count', None)
            rep = int(st.get('report_every', 100) or 100)
            nxt = int(st.get('next_report', rep) or rep)

            if extracted == 1:
                msg = f'Frame queue: started {video_path.name}'
            elif extracted >= nxt:
                if fc2:
                    pct = int((extracted / int(fc2)) * 100)
                    msg = f'Frame queue: {video_path.name} {extracted}/{int(fc2)} ({pct}%)'
                else:
                    msg = f'Frame queue: {video_path.name} extracted {extracted}'
                st['next_report'] = int(nxt) + int(rep)

            if fc2 and extracted >= int(fc2):
                st['done'] = True
                msg = f'Frame queue: done {video_path.name} ({extracted}/{int(fc2)})'

    if msg:
        web_print(msg)

    return task


def _fill_frame_queue(max_new: int = 200) -> int:
    try:
        max_new = int(max_new)
    except Exception:
        max_new = 200
    max_new = max(1, min(max_new, 2000))

    filled = 0
    with FRAME_QUEUE_FILL_LOCK:
        while filled < max_new:
            with FRAME_QUEUE_LOCK:
                max_len = FRAME_QUEUE.maxlen or 0
                if max_len and len(FRAME_QUEUE) >= max_len:
                    break

            with FRAME_VIDEO_STATE_LOCK:
                if not FRAME_VIDEO_ORDER:
                    break
                key = FRAME_VIDEO_ORDER.popleft()

            try:
                task = _next_frame_task_for_video(key)
            except Exception as exc:
                with FRAME_VIDEO_STATE_LOCK:
                    FRAME_VIDEO_STATE.pop(key, None)
                web_print(f'Frame queue: dropped video due to error: {exc}')
                continue

            if not task:
                with FRAME_VIDEO_STATE_LOCK:
                    FRAME_VIDEO_STATE.pop(key, None)
                continue

            with FRAME_QUEUE_LOCK:
                max_len = FRAME_QUEUE.maxlen or 0
                if (not max_len) or len(FRAME_QUEUE) < max_len:
                    FRAME_QUEUE.append(task)
                    filled += 1

            with FRAME_VIDEO_STATE_LOCK:
                st = FRAME_VIDEO_STATE.get(key)
                if st and not st.get('done', False):
                    FRAME_VIDEO_ORDER.append(key)
                else:
                    FRAME_VIDEO_STATE.pop(key, None)

    return int(filled)


def _frame_queue_worker_loop():
    web_print('Frame queue worker started')
    while True:
        if FRAME_QUEUE_STOP_EVENT.is_set():
            break
        try:
            filled = _fill_frame_queue(max_new=200)
        except Exception as exc:
            filled = 0
            web_print(f'Frame queue worker error: {exc}')
        if filled <= 0:
            FRAME_QUEUE_WAKE_EVENT.wait(timeout=0.5)
            FRAME_QUEUE_WAKE_EVENT.clear()


def start_frame_queue_worker():
    global FRAME_QUEUE_WORKER
    try:
        if FRAME_QUEUE_WORKER is not None and FRAME_QUEUE_WORKER.is_alive():
            return
    except Exception:
        pass
    try:
        FRAME_QUEUE_STOP_EVENT.clear()
    except Exception:
        pass
    FRAME_QUEUE_WORKER = Thread(target=_frame_queue_worker_loop, daemon=True, name='Frame-Queue-Worker')
    FRAME_QUEUE_WORKER.start()


def da3_task_queue_peek(limit: int = 200):
    try:
        limit = int(limit)
    except Exception:
        limit = 200
    limit = max(1, min(limit, 2000))

    with FRAME_QUEUE_LOCK:
        items = list(itertools.islice(reversed(FRAME_QUEUE), limit))
    items.reverse()
    return items


def da3_task_queue_pop():
    with FRAME_QUEUE_LOCK:
        if not FRAME_QUEUE:
            return None
        item = FRAME_QUEUE.popleft()
    FRAME_QUEUE_WAKE_EVENT.set()
    return item


def enqueue_video(path: Path, source: str = "") -> None:
    original_path = path
    if path.suffix.lower() == ".dav":
        web_print(f"Converting DAV to MP4: {path.name}")
        try:
            path = convert_dav_to_mp4(path)
        except Exception as exc:
            web_print(f"DAV->MP4 conversion failed: {exc}")
            return
        web_print(f"Converted DAV -> MP4: {path.name}")
        source = (source + " (dav->mp4)") if source else "dav->mp4"

    if not _is_video_path(path):
        web_print(f"[enqueue_video] : {path} is not a video path")
        return

    try:
        size_bytes = path.stat().st_size
    except Exception:
        size_bytes = 0

    item = {
        "stored_name": path.name,
        "path": str(path),
        "size_bytes": int(size_bytes),
        "size_mb": round(size_bytes / (1024 * 1024), 2),
        "ts": float(time.time()),
        "source": str(source or ""),
        "original_stored_name": original_path.name if original_path != path else "",
        "original_path": str(original_path) if original_path != path else "",
    }

    dropped = None
    with VIDEO_QUEUE_LOCK:
        if VIDEO_UPLOAD_QUEUE.maxlen and len(VIDEO_UPLOAD_QUEUE) >= VIDEO_UPLOAD_QUEUE.maxlen:
            dropped = VIDEO_UPLOAD_QUEUE[0]
        VIDEO_UPLOAD_QUEUE.append(item)

    if dropped:
        web_print(f"Video queue full; dropped oldest: {dropped.get('stored_name')}")

    src = f" ({source})" if source else ""
    web_print(f"Enqueued video{src}: {item['stored_name']} ({item['size_mb']} MB)")

    try:
        enqueue_da3_job(path, source=source)
    except Exception as exc:
        web_print(f"FPS adjust enqueue failed for {path.name}: {exc}")


def video_queue_list():
    with VIDEO_QUEUE_LOCK:
        return list(VIDEO_UPLOAD_QUEUE)


def video_queue_pop():
    with VIDEO_QUEUE_LOCK:
        if not VIDEO_UPLOAD_QUEUE:
            return None
        return VIDEO_UPLOAD_QUEUE.popleft()


def _is_port_free(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.2)
        return sock.connect_ex((host, port)) != 0


def _pick_port(host: str, preferred: int = 5000) -> int:
    if _is_port_free(host, preferred):
        return preferred
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((host, 0))
        return int(sock.getsockname()[1])


class _ServerThread(Thread):
    def __init__(self, app: Flask, host: str, port: int):
        super().__init__(daemon=True)
        self._server = make_server(host, port, app)
        self._ctx = app.app_context()
        self._ctx.push()

    def run(self):
        self._server.serve_forever()

    def shutdown(self):
        self._server.shutdown()


# If you re-run this cell, stop the previous server first.
if "server" in globals():
    try:
        server.shutdown()
    except Exception:
        pass

# Persist kernel start time across cell re-runs.
if "_KERNEL_START_TS" not in globals():
    _KERNEL_START_TS = _get_process_start_time_epoch() or time.time()
KERNEL_START_TS = float(_KERNEL_START_TS)

# Persist settings across cell re-runs.
if "SETTINGS_LOCK" not in globals():
    SETTINGS_LOCK = Lock()
if "_TARGET_FPS" not in globals():
    _TARGET_FPS = 15


def get_target_fps() -> int:
    with SETTINGS_LOCK:
        try:
            return int(_TARGET_FPS)
        except Exception:
            return 15


def set_target_fps(value) -> int:
    global _TARGET_FPS
    fps = int(value)
    if fps < 1 or fps > 240:
        raise ValueError("Target FPS must be between 1 and 240.")
    with SETTINGS_LOCK:
        _TARGET_FPS = fps
    return fps


app = Flask(__name__)
UPLOAD_DIR = Path("outputs/uploads")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR = UPLOAD_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
start_da3_worker()
start_frame_queue_worker()

# On cell re-runs, also prepare already-queued videos.
try:
    with VIDEO_QUEUE_LOCK:
        _existing = list(VIDEO_UPLOAD_QUEUE)
    for _it in _existing:
        try:
            _p = Path(_it.get('path', ''))
            if _p and _p.exists():
                enqueue_da3_job(_p, source=_it.get('source', ''))
        except Exception:
            pass
except Exception:
    pass

# On cell re-runs, also build tasks for already FPS-adjusted videos.
try:
    with DA3_QUEUE_LOCK:
        _fps_existing = list(DA3_QUEUE)
    for _it in _fps_existing:
        try:
            _p = Path(_it.get('path', ''))
            if _p and _p.exists():
                register_video_for_frame_queue(_p, source=_it.get('source', ''))
        except Exception:
            pass
except Exception:
    pass
app.config["MAX_CONTENT_LENGTH"] = 25 * 1024 * 1024 * 1024  # 25GB

_HOME_HTML = """
<!doctype html>
<title>Flask + ngrok demo</title>
<style>
body { font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; margin: 20px; }
#terminal {
  background: #0b0f0b;
  color: #d1f7c4;
  border: 1px solid #1f2a1f;
  border-radius: 8px;
  padding: 12px;
  height: 360px;
  overflow-y: auto;
  white-space: pre-wrap;
  font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;
  font-size: 13px;
  line-height: 1.35;
}
.small { color: #666; font-size: 12px; }
</style>
<h1>Flask + ngrok demo</h1>

<p><b>Kernel uptime:</b> {{ uptime_hhmm }} (HH:mm)</p>
<p><b>Remaining to 12h:</b> {{ remaining_hhmm }} (HH:mm)</p>

<h2>Settings</h2>
<form method=\"post\" action=\"{{ url_for('set_target_fps_route') }}\">
  <label for=\"target_fps\"><b>Target FPS:</b></label>
  <input type=\"number\" id=\"target_fps\" name=\"target_fps\" min=\"1\" max=\"240\" step=\"1\" value=\"{{ target_fps }}\" required>
  <button type=\"submit\">Set</button>
</form>

<h2>Download by URL</h2>
<form method=\"post\" action=\"{{ url_for('upload_url') }}\">
  <label for=\"method\"><b>Method:</b></label>
  <select name=\"method\" id=\"method\" onchange=\"updateUrlMethodUI()\">
    <option value=\"wget\" {% if method == 'wget' %}selected{% endif %}>wget</option>
    <option value=\"google-drive\" {% if method == 'google-drive' %}selected{% endif %}>google-drive</option>
    <option value=\"mega.nz\" {% if method == 'mega.nz' %}selected{% endif %}>mega.nz</option>
  </select>
  <br>
  <input type=\"url\" name=\"url\" placeholder=\"https://example.com/file.jpg\" value=\"{{ url_value }}\" style=\"width: 420px\" required>
  <div id=\"filename_row\" style=\"margin-top: 6px; display: none;\">
    <input type=\"text\" name=\"filename\" id=\"filename\" placeholder=\"output filename (required for google-drive/mega.nz)\" value=\"{{ filename_value }}\" style=\"width: 420px\">
  </div>
  <div style=\"margin-top: 6px;\">
    <button type=\"submit\">Download</button>
  </div>
</form>

<script>
function updateUrlMethodUI() {
  var method = document.getElementById('method');
  var row = document.getElementById('filename_row');
  var input = document.getElementById('filename');
  if (!method || !row || !input) return;
  var needs = (method.value === 'google-drive' || method.value === 'mega.nz');
  row.style.display = needs ? 'block' : 'none';
  input.required = needs;
  input.disabled = !needs;
}
async function refreshTerminal() {
  var el = document.getElementById('terminal');
  if (!el) return;
  try {
    var res = await fetch('/api/terminal');
    var data = await res.json();
    var shouldStick = (el.scrollTop + el.clientHeight + 30) >= el.scrollHeight;
    el.textContent = data.text || '';
    if (shouldStick) { el.scrollTop = el.scrollHeight; }
  } catch (e) {
    el.textContent = 'Error loading terminal: ' + e;
  }
}
document.addEventListener('DOMContentLoaded', function () {
  updateUrlMethodUI();
  refreshTerminal();
  setInterval(refreshTerminal, 1000);
});
</script>

<h2>Upload a file</h2>
<form method=\"post\" action=\"{{ url_for('upload_file') }}\" enctype=\"multipart/form-data\">
  <input type=\"file\" name=\"file\" required>
  <button type=\"submit\">Upload file</button>
</form>

<h2>Video queue</h2>
<p><b>Queued videos:</b> {{ video_queue_len }}{% if video_queue_max %} / {{ video_queue_max }}{% endif %}</p>
{% if video_queue_len %}
  <ol>
  {% for item in video_queue %}
    <li><a href=\"{{ url_for('get_upload', filename=item['stored_name']) }}\">{{ item['stored_name'] }}</a> ({{ item['size_mb'] }} MB)</li>
  {% endfor %}
  </ol>
{% else %}
  <p class=\"small\">No videos queued yet.</p>
{% endif %}

<h2>fps adjusted queue</h2>
<p><b>FPS adjusted videos:</b> {{ fps_queue_len }}{% if fps_queue_max %} / {{ fps_queue_max }}{% endif %} (pending jobs: {{ fps_jobs_pending }})</p>
{% if fps_queue_len %}
  <ol>
  {% for item in fps_queue %}
    <li><a href=\"{{ url_for('get_upload', filename=item['stored_name']) }}\">{{ item['stored_name'] }}</a> ({{ item['size_mb'] }} MB, {{ item['target_fps'] }} fps)</li>
  {% endfor %}
  </ol>
{% else %}
  <p class=\"small\">No FPS adjusted videos yet.</p>
{% endif %}

<h2>Frame queue (DA3 tasks)</h2>
<p><b>DA3 ready:</b> {{ da3_ready }} | <b>Tasks:</b> {{ da3_task_queue_len }}{% if da3_task_queue_max %} / {{ da3_task_queue_max }}{% endif %} (pending videos: {{ da3_task_pending_videos }})</p>
<p class=\"small\">Peek: <a href=\"/api/da3_task_queue\">/api/da3_task_queue</a></p>

<hr>
<ul>
  <li><a href=\"/api/ping\">/api/ping</a></li>
  <li><a href=\"/api/echo?msg=hello\">/api/echo?msg=hello</a></li>
</ul>

{% if message %}
  <p><b>{{ message }}</b></p>
{% endif %}

{% if uploaded_url %}
  <p>Saved: <a href=\"{{ uploaded_url }}\">{{ uploaded_url }}</a></p>
{% endif %}

<h2>Web terminal</h2>
<p class=\"small\">Call <code>web_print('hello')</code> from notebook cells to print here.</p>
<pre id=\"terminal\">{{ terminal_text|e }}</pre>
"""


def _home_context(**kwargs):
    elapsed = time.time() - KERNEL_START_TS
    _drain_web_print_queue(max_items=2000)
    with WEB_TERMINAL_LOCK:
        terminal_text = "\n".join(WEB_TERMINAL_BUFFER)
    with VIDEO_QUEUE_LOCK:
        video_queue = list(VIDEO_UPLOAD_QUEUE)
    with DA3_QUEUE_LOCK:
        fps_queue = list(DA3_QUEUE)
    try:
        fps_jobs_pending = DA3_JOB_QUEUE.qsize()
    except Exception:
        fps_jobs_pending = 0
    with FRAME_QUEUE_LOCK:
        da3_task_queue_len = len(FRAME_QUEUE)
        da3_task_queue_max = FRAME_QUEUE.maxlen or 0
    with FRAME_VIDEO_STATE_LOCK:
        da3_task_pending_videos = len(FRAME_VIDEO_ORDER)
    ctx = {
        "uptime_hhmm": _format_hhmm(elapsed),
        "remaining_hhmm": _format_hhmm(TWELVE_HOURS_SEC - elapsed),
        "method": "wget",
        "url_value": "",
        "filename_value": "",
        "terminal_text": terminal_text,
        "target_fps": get_target_fps(),
        "video_queue_len": len(video_queue),
        "video_queue_max": VIDEO_UPLOAD_QUEUE.maxlen or 0,
        "video_queue": video_queue,
        "fps_queue_len": len(fps_queue),
        "fps_queue_max": DA3_QUEUE.maxlen or 0,
        "fps_queue": fps_queue,
        "fps_jobs_pending": fps_jobs_pending,
        "da3_ready": _is_da3_set(),
        "da3_task_queue_len": da3_task_queue_len,
        "da3_task_queue_max": da3_task_queue_max,
        "da3_task_pending_videos": da3_task_pending_videos,
    }
    ctx.update(kwargs)
    return ctx


MAX_URL_DOWNLOAD_BYTES = 25 * 1024 * 1024 * 1024  # 25GB
MAX_URL_REDIRECTS = 3
ALLOWED_URL_METHODS = {'wget', 'google-drive', 'mega.nz'}


def _hostname_looks_public(hostname: str) -> bool:
    try:
        infos = socket.getaddrinfo(hostname, None)
    except socket.gaierror:
        return False

    for info in infos:
        ip_str = info[4][0]
        try:
            ip = ipaddress.ip_address(ip_str)
        except ValueError:
            return False

        if (
            ip.is_private
            or ip.is_loopback
            or ip.is_link_local
            or ip.is_reserved
            or ip.is_multicast
            or ip.is_unspecified
        ):
            return False

    return True


def _validate_fetch_url(parsed: urllib.parse.ParseResult) -> None:
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError("URL must start with http:// or https://")
    if parsed.username or parsed.password:
        raise ValueError("URLs with embedded credentials are not allowed.")
    if not parsed.hostname:
        raise ValueError("Invalid URL hostname.")

    try:
        port = parsed.port
    except ValueError as exc:
        raise ValueError("Invalid port in URL.") from exc
    if port not in {None, 80, 443}:
        raise ValueError("Only ports 80 and 443 are allowed.")

    if not _hostname_looks_public(parsed.hostname):
        raise ValueError("Refusing to fetch from private/reserved hosts.")


class _SafeRedirect(urllib.request.HTTPRedirectHandler):
    def __init__(self, max_redirects: int):
        super().__init__()
        self._remaining = int(max_redirects)

    def redirect_request(self, req, fp, code, msg, headers, newurl):
        self._remaining -= 1
        if self._remaining < 0:
            raise ValueError("Too many redirects.")

        if not urllib.parse.urlparse(newurl).scheme:
            newurl = urllib.parse.urljoin(req.full_url, newurl)

        _validate_fetch_url(urllib.parse.urlparse(newurl))
        return super().redirect_request(req, fp, code, msg, headers, newurl)


def handle_url_upload(method: str, url: str, filename: str = "") -> Path:
    method = (method or "wget").strip()
    if method not in ALLOWED_URL_METHODS:
        raise ValueError(f"Unknown method: {method}")

    url = (url or "").strip()
    if not url:
        raise ValueError("Please enter a URL.")
    if len(url) > 2048:
        raise ValueError("URL is too long.")

    parsed = urllib.parse.urlparse(url)
    _validate_fetch_url(parsed)

    safe_filename = secure_filename((filename or "").strip())
    if safe_filename and len(safe_filename) > 150:
        raise ValueError("Filename is too long.")

    # Keep MEGA fragments (#key); other methods don't need fragments.
    cleaned = url if method == "mega.nz" else parsed._replace(fragment="").geturl()

    if method == "google-drive":
        host = (parsed.hostname or "").lower()
        if not (host.endswith("drive.google.com") or host.endswith("docs.google.com")):
            raise ValueError("Expected a Google Drive URL (drive.google.com / docs.google.com).")
        if not safe_filename:
            raise ValueError("Filename is required for google-drive.")

        out_name = f"gdrive_{uuid.uuid4().hex[:12]}_{safe_filename}"
        out_path = UPLOAD_DIR / out_name

        try:
            import gdown  # type: ignore
        except Exception as exc:
            raise ValueError("gdown is not installed. Run the pip install cell.") from exc

        result = None
        try:
            result = gdown.download(cleaned, str(out_path), quiet=True, fuzzy=True)
        except TypeError:
            result = gdown.download(cleaned, str(out_path), quiet=True)

        if not out_path.exists():
            if result and Path(str(result)).exists():
                Path(str(result)).replace(out_path)
            else:
                raise ValueError("Google Drive download failed.")

        if out_path.stat().st_size > MAX_URL_DOWNLOAD_BYTES:
            out_path.unlink(missing_ok=True)
            raise ValueError(f"Downloaded file too large (>{MAX_URL_DOWNLOAD_BYTES} bytes).")

        enqueue_video(out_path, source="url:google-drive")
        return out_path

    if method == "mega.nz":
        host = (parsed.hostname or "").lower()
        if not (host.endswith("mega.nz") or host.endswith("mega.co.nz")):
            raise ValueError("Expected a mega.nz URL.")
        if not safe_filename:
            raise ValueError("Filename is required for mega.nz.")

        out_name = f"mega_{uuid.uuid4().hex[:12]}_{safe_filename}"
        out_path = UPLOAD_DIR / out_name

        try:
            from mega import Mega  # type: ignore
        except Exception as exc:
            raise ValueError("mega.py is not installed. Run the pip install cell.") from exc

        client = Mega().login()
        downloaded = None
        for fn in (
            lambda: client.download_url(cleaned, dest_path=str(UPLOAD_DIR), dest_filename=out_path.name),
            lambda: client.download_url(cleaned, str(UPLOAD_DIR), out_path.name),
            lambda: client.download_url(cleaned, str(UPLOAD_DIR)),
        ):
            try:
                downloaded = fn()
                break
            except TypeError:
                continue

        if not out_path.exists() and downloaded:
            downloaded_path = Path(str(downloaded))
            if downloaded_path.exists():
                try:
                    downloaded_path.replace(out_path)
                except Exception:
                    pass

        if not out_path.exists():
            raise ValueError("MEGA download failed.")

        if out_path.stat().st_size > MAX_URL_DOWNLOAD_BYTES:
            out_path.unlink(missing_ok=True)
            raise ValueError(f"Downloaded file too large (>{MAX_URL_DOWNLOAD_BYTES} bytes).")

        enqueue_video(out_path, source="url:mega.nz")
        return out_path

    # wget
    guessed_name = safe_filename or Path(urllib.parse.unquote(parsed.path or "")).name
    guessed_name = secure_filename(guessed_name) or "download.bin"
    out_name = f"wget_{uuid.uuid4().hex[:12]}_{guessed_name}"
    out_path = UPLOAD_DIR / out_name

    opener = urllib.request.build_opener(_SafeRedirect(MAX_URL_REDIRECTS))
    req = urllib.request.Request(cleaned, headers={"User-Agent": "Flask-ngrok-demo/1.0"})

    total = 0
    try:
        with opener.open(req, timeout=10) as resp, open(out_path, "wb") as f:
            while True:
                chunk = resp.read(64 * 1024)
                if not chunk:
                    break
                total += len(chunk)
                if total > MAX_URL_DOWNLOAD_BYTES:
                    raise ValueError(f"Remote file too large (>{MAX_URL_DOWNLOAD_BYTES} bytes).")
                f.write(chunk)
    except Exception:
        try:
            out_path.unlink(missing_ok=True)
        except Exception:
            pass
        raise

    enqueue_video(out_path, source="url:wget")
    return out_path


def handle_file_upload(file_storage) -> Path:
    if file_storage is None or not getattr(file_storage, "filename", ""):
        raise ValueError("Please choose a file.")

    original_name = secure_filename(file_storage.filename) or "upload.bin"
    out_name = f"file_{uuid.uuid4().hex[:12]}_{original_name}"
    out_path = UPLOAD_DIR / out_name
    file_storage.save(out_path)
    enqueue_video(out_path, source="file-upload")
    return out_path


@app.get("/")
def home():
    return render_template_string(_HOME_HTML, **_home_context())


@app.post("/settings/target_fps")
def set_target_fps_route():
    raw = request.form.get("target_fps", "")
    try:
        fps = set_target_fps(raw)
    except Exception as exc:
        return render_template_string(_HOME_HTML, **_home_context(message=f"Invalid Target FPS: {exc}")), 400

    web_print(f"Target FPS set to {fps}")
    return render_template_string(_HOME_HTML, **_home_context(message=f"Target FPS set to {fps}"))


@app.get("/api/terminal")
def terminal_api():
    _drain_web_print_queue(max_items=2000)
    with WEB_TERMINAL_LOCK:
        text = "\n".join(WEB_TERMINAL_BUFFER)
    return jsonify(ok=True, text=text, lines=len(WEB_TERMINAL_BUFFER))


@app.get("/api/video_queue")
def video_queue_api():
    with VIDEO_QUEUE_LOCK:
        items = list(VIDEO_UPLOAD_QUEUE)
        max_len = VIDEO_UPLOAD_QUEUE.maxlen or 0
    return jsonify(ok=True, count=len(items), max=max_len, items=items)


@app.post("/api/video_queue/pop")
def video_queue_pop_api():
    item = video_queue_pop()
    if item:
        web_print(f"Dequeued video: {item.get('stored_name')}")
    return jsonify(ok=True, item=item)


@app.post("/api/video_queue/clear")
def video_queue_clear_api():
    with VIDEO_QUEUE_LOCK:
        VIDEO_UPLOAD_QUEUE.clear()
    web_print("Video queue cleared.")
    return jsonify(ok=True)


@app.get("/api/fps_adjusted_queue")
@app.get("/api/da3_queue")
def fps_adjusted_queue_api():
    with DA3_QUEUE_LOCK:
        items = list(DA3_QUEUE)
        max_len = DA3_QUEUE.maxlen or 0
    try:
        pending = DA3_JOB_QUEUE.qsize()
    except Exception:
        pending = 0
    return jsonify(ok=True, count=len(items), max=max_len, pending=pending, items=items)


@app.post("/api/fps_adjusted_queue/pop")
@app.post("/api/da3_queue/pop")
def fps_adjusted_queue_pop_api():
    item = da3_queue_pop()
    if item:
        web_print(f"Dequeued FPS adjusted video: {item.get('stored_name')}")
    return jsonify(ok=True, item=item)


@app.post("/api/fps_adjusted_queue/clear")
@app.post("/api/da3_queue/clear")
def fps_adjusted_queue_clear_api():
    with DA3_QUEUE_LOCK:
        DA3_QUEUE.clear()
    web_print("FPS adjusted queue cleared.")
    return jsonify(ok=True)


@app.get("/api/da3_task_queue")
def da3_task_queue_api():
    limit = request.args.get("limit", "200")
    items = da3_task_queue_peek(limit)
    with FRAME_QUEUE_LOCK:
        count = len(FRAME_QUEUE)
        max_len = FRAME_QUEUE.maxlen or 0
    with FRAME_VIDEO_STATE_LOCK:
        pending_videos = len(FRAME_VIDEO_ORDER)
    return jsonify(ok=True, da3_ready=_is_da3_set(), count=count, max=max_len, pending_videos=pending_videos, items=items)


@app.post("/api/da3_task_queue/pop")
def da3_task_queue_pop_api():
    item = da3_task_queue_pop()
    if item:
        web_print(f"Dequeued DA3 task: {item.get('video_file_url')} frame {item.get('frame_no')}")
    return jsonify(ok=True, item=item)


@app.post("/api/da3_task_queue/clear")
def da3_task_queue_clear_api():
    with FRAME_QUEUE_LOCK:
        FRAME_QUEUE.clear()
    with FRAME_VIDEO_STATE_LOCK:
        FRAME_VIDEO_STATE.clear()
        FRAME_VIDEO_ORDER.clear()
    FRAME_QUEUE_WAKE_EVENT.set()
    web_print("Frame queue cleared.")
    return jsonify(ok=True)


@app.post("/upload/url")
def upload_url():
    method = request.form.get("method", "wget")
    url = request.form.get("url", "")
    filename = request.form.get("filename", "")

    try:
        saved_path = handle_url_upload(method, url, filename)
    except Exception as exc:
        return render_template_string(
            _HOME_HTML,
            **_home_context(
                message=f"URL fetch failed: {exc}",
                method=method,
                url_value=url,
                filename_value=filename,
            ),
        ), 400

    return render_template_string(
        _HOME_HTML,
        **_home_context(
            message=f"URL downloaded via {method}.",
            uploaded_url=url_for("get_upload", filename=saved_path.name),
            method=method,
            url_value=url,
            filename_value=filename,
        ),
    )


@app.post("/upload/file")
def upload_file():
    try:
        saved_path = handle_file_upload(request.files.get("file"))
    except Exception as exc:
        return render_template_string(_HOME_HTML, **_home_context(message=f"File upload failed: {exc}")), 400

    return render_template_string(
        _HOME_HTML,
        **_home_context(
            message="File uploaded.",
            uploaded_url=url_for("get_upload", filename=saved_path.name),
        ),
    )


@app.get("/uploads/<path:filename>")
def get_upload(filename: str):
    return send_from_directory(str(UPLOAD_DIR), filename)


@app.get("/frames/<path:filename>")
def get_frame(filename: str):
    return send_from_directory(str(FRAMES_DIR), filename)


@app.errorhandler(413)
def too_large(_exc):
    return render_template_string(_HOME_HTML, **_home_context(message="Upload too large (max 25GB).")), 413


@app.get("/api/ping")
def ping():
    return jsonify(ok=True, message="pong", ts=time.time())


@app.get("/api/echo")
def echo():
    return jsonify(ok=True, msg=request.args.get("msg", ""))


HOST = "127.0.0.1"
PORT = _pick_port(HOST, preferred=5000)

server = _ServerThread(app, HOST, PORT)
server.start()

web_print(f"Flask running locally: http://{HOST}:{PORT}")
print(f"Flask running locally: http://{HOST}:{PORT}")

In [5]:
import os
from pyngrok import ngrok as ngrok_api

# If you re-run this cell, close old tunnels first.
try:
    ngrok_api.kill()
except Exception:
    pass

# Option A: set an env var before launching Jupyter (recommended)
#   Windows (PowerShell): $env:NGROK_AUTHTOKEN="..."
#   macOS/Linux (bash):  export NGROK_AUTHTOKEN="..."
# Option B: paste your token here (not recommended for shared notebooks)
token = os.environ.get("NGROK_AUTHTOKEN", "").strip() or ""

if token:
    ngrok_api.set_auth_token(token)

tunnel = ngrok_api.connect(PORT)
public_url = tunnel.public_url

print("ngrok public URL:", public_url)
print("Try:", public_url + "/api/ping")

# Conda Setup

In [6]:
web_print("Setting up conda")

In [7]:
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!chmod +x miniconda.sh
!bash miniconda.sh -b -p /miniconda

!. /miniconda/etc/profile.d/conda.sh && conda init
!. /miniconda/etc/profile.d/conda.sh && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!. /miniconda/etc/profile.d/conda.sh && conda create -n my_env python=3.12 -y && conda activate my_env

In [8]:
%%writefile on_conda
#!/bin/bash

src=". /miniconda/etc/profile.d/conda.sh"
env="conda activate my_env"
cmd="$src && $env && $@"
eval $cmd

In [ ]:
!chmod -x on_conda

import os
os.environ["konda"] = "bash on_conda"
!$konda conda --version

isCondaSet = True

In [10]:
if isCondaSet:
    web_print("Conda is installed succesfully.")
else:
    web_print("Unable to set conda")

# DA3 Setup

In [ ]:
web_print("Setting up DA3")

In [12]:
!git clone https://github.com/ByteDance-Seed/Depth-Anything-3.git

In [ ]:
!$konda pip install xformers torch\>=2 torchvision -q
!$konda pip install -e Depth-Anything-3/. -q # Basic 
!$konda pip install --no-build-isolation git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70 -q# for gaussian head
!$konda pip install -e "Depth-Anything-3/.[app]" -q # Gradio, python>=3.10
!$konda pip install -e "Depth-Anything-3/.[all]" -q # ALL

In [ ]:
# Install required packages
!pip install pyngrok fastapi uvicorn -q
isDA3Set = True

In [ ]:
if isDA3Set:
    web_print("DA3 is installed succesfully.")
else:
    web_print("Unable to set DA3")

# Stop / cleanup

Run the next cell when you're done to stop ngrok and the Flask server.

In [ ]:
# Stop ngrok (if running)
try:
    from pyngrok import ngrok as ngrok_api

    if "public_url" in globals():
        try:
            ngrok_api.disconnect(public_url)
        except Exception:
            pass
    ngrok_api.kill()
except Exception:
    pass

# Stop Flask (if running)
if "server" in globals():
    try:
        server.shutdown()
    except Exception:
        pass

print("Stopped ngrok + Flask.")

# Check upload directory

In [ ]:
!ls /kaggle/working/outputs/uploads/frames/fpsadj_990e512928b1_gdrive_07a83daa7d55_08.15.01-08.20.00R000_fps15